In [0]:
%run ./01-config

In [0]:
test_data_dir = base_dir_data + "/test-data"


In [0]:
#--------------------------------------------------------------------
# LOAD HISTORY
#--------------------------------------------------------------------

# One pre-existing table that needs uploading - date_lookup. 
# Possible to add more if required
# date_lookup is a known, good historical data
# date_lookup is a static, dimension table 
# every time this script is run you want it to end up with the exact same table and data (idempotency) 

#--------------------------------------------------------------------
# Load table function(s)
#--------------------------------------------------------------------

# USE INSERT OVERWRITE here to ensure the table is the same every time 
# (eg. INSERT INTO would append, so every time this script is run this table would be altered)
# This will wipe it clear and replace with the new data

def load_date_lookup(catalog, db_name):
    print(f"Creating date_lookup table...", end='')
    spark.sql(f"""INSERT OVERWRITE TABLE {catalog}.{db_name}.date_lookup    -- idempotency
            Select 
                date, 
                week, 
                year, 
                month, 
                dayofweek, 
                dayofmonth, 
                dayofyear, 
                week_part
            FROM json.`{test_data_dir}/6-date-lookup.json/`""")
    print("Done")

    #abfss://sbit-unmanaged-dev@sarahlarkin.dfs.core.windows.net/data-zone/test_data/6-date-lookup.json

# if there were more tables to be loaded you would create separate functions here 

#--------------------------------------------------------------------
# Call the load table function(s)
#--------------------------------------------------------------------

def load_history(catalog, db_name):
    """ - create start time & print statment for start 
        - call the function(s):  load_date_lookup(catalog, db_name)
        - print the total time taken """
    import time
    start = int(time.time())
    print(f"\nStarting historical data load ...")
    load_date_lookup(catalog, db_name) 
    # if other historical tables were needed you would call them here
    print(f"Historical data load completed in {int(time.time()) - start} seconds")

#--------------------------------------------------------------------
# VALIDATE
#--------------------------------------------------------------------

# validate by: checking no. of rows uploaded is the same as expected 
def assert_count(catalog, db_name, table_name, expected_count):  #table_name and expected_count defined in validate() below 
    """ - read the table to count actual number of records returned 
        - assert the actual == expected_count (arg) 
        - return error msg if fails / print success message """

    print(f"Validating record counts in {table_name}...", end='')
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").count()
    assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}" # ":," adds commas to numbers
    #assert <condition>, <optional error message> (msg only executed in condition is False)
    print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

#call assert_count() func passing in table_name and expected_count and record execution time 
def validate(catalog, db_name):
    """ - create start time & print statment for start 
        - call the function: assert_count(catalog, db_name, table_name, expected_count)
        - print the total time taken """

    import time
    start = int(time.time())
    print(f"\nStarting historical data load validation...")
    assert_count(catalog, db_name, "date_lookup", 365)      # passing in table_name and expected_count --> check the files to find out the expected values 
    print(f"Historical data load validation completed in {int(time.time()) - start} seconds")
